In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/images/test_02_test.tif
/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/images/test_04_test.tif
/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/images/test_03_test.tif
/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/images/test_08_test.tif
/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/images/test_01_test.tif
/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/images/test_06_test.tif
/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/images/test_07_test.tif
/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/images/test_05_test.tif
/kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/CHASE/test/masks/test_04_manual1.tif
/kaggle/input/datasets/abhinav2807/retinal66

In [3]:
!pip install albumentations thop medpy scikit-image -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 1.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [ ]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

import os
import re
import csv
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    matthews_corrcoef
)
from tqdm.auto import tqdm


# ---------------- CONFIGURATION ----------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_PATH = "/kaggle/input/datasets/abhinav2807/retinal66"
OUTPUT_ROOT = "/kaggle/working/stare_results"

IMG_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 100
LEARNING_RATE = 2e-4
NUM_WORKERS = 2

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Device:", device)
print("Dataset root:", BASE_PATH)
print("Output folder:", OUTPUT_ROOT)


# ============================================================
# 2. FIND THE ACTUAL STARE DIRECTORY
# ============================================================

def find_stare_root(dataset_root):
    """
    Locate the STARE directory inside the supplied Kaggle path.
    If no STARE directory exists, use the supplied path itself.
    """

    if not os.path.isdir(dataset_root):
        raise FileNotFoundError(
            f"Dataset path does not exist:\n{dataset_root}\n\n"
            "Check the Kaggle dataset attachment and path."
        )

    if os.path.basename(os.path.normpath(dataset_root)).lower() == "stare":
        return dataset_root

    exact_matches = []
    partial_matches = []

    for current, dirs, files in os.walk(dataset_root):
        folder_name = os.path.basename(current).lower()

        if folder_name == "stare":
            exact_matches.append(current)
        elif "stare" in folder_name:
            partial_matches.append(current)

    if exact_matches:
        exact_matches.sort(key=lambda p: (p.count(os.sep), len(p)))
        return exact_matches[0]

    if partial_matches:
        partial_matches.sort(key=lambda p: (p.count(os.sep), len(p)))
        return partial_matches[0]

    return dataset_root


# ============================================================
# 3. AUTOMATIC IMAGE AND MASK DISCOVERY
# ============================================================

IMAGE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
    ".ppm",
    ".pgm"
}

MASK_DIR_KEYWORDS = (
    "mask",
    "label",
    "groundtruth",
    "ground_truth",
    "segmentation",
    "vessel",
    "manual",
    "annotation"
)

MASK_SUFFIX_PATTERN = re.compile(
    r"(?:[._-](?:ah|vk|mask|masks|label|labels|vessel|seg|"
    r"segmentation|gt|groundtruth|manual\d*|annotation\d*|"
    r"1st_manual|2nd_manual))+$",
    re.IGNORECASE
)


def filename_without_image_extensions(path):
    """
    Remove image extensions while supporting filenames such as
    im0001.ah.ppm and im0001.vk.ppm.
    """

    name = os.path.basename(path).lower()

    while True:
        stem, ext = os.path.splitext(name)

        if ext.lower() in IMAGE_EXTENSIONS:
            name = stem
        else:
            break

    return name


def canonical_key(path):
    """
    Generate a common key for an image and its corresponding mask.

    Examples:
        im0001.ppm     -> im0001
        im0001.ah.ppm  -> im0001
        im0001.vk.ppm  -> im0001
        im0001_mask.png -> im0001
    """

    name = filename_without_image_extensions(path)

    previous = None

    while previous != name:
        previous = name
        name = MASK_SUFFIX_PATTERN.sub("", name)

    name = re.sub(r"[^a-z0-9]", "", name)

    return name


def is_mask_file(path):
    """
    Detect masks using their directory name or filename suffix.
    """

    parent = os.path.dirname(path)
    parent_parts = [
        part.lower()
        for part in os.path.normpath(parent).split(os.sep)
    ]

    for part in parent_parts:
        if any(keyword in part for keyword in MASK_DIR_KEYWORDS):
            return True

    stem = filename_without_image_extensions(path)

    if MASK_SUFFIX_PATTERN.search(stem):
        return True

    return False


def mask_priority(path):
    """
    Prefer the AH annotation when multiple STARE annotations exist.
    """

    lower_path = path.lower()
    filename = filename_without_image_extensions(path)

    if "labels-ah" in lower_path or re.search(r"[._-]ah$", filename):
        return 0

    if "manual1" in lower_path or "1st_manual" in lower_path:
        return 1

    if "labels-vk" in lower_path or re.search(r"[._-]vk$", filename):
        return 2

    return 3


def display_directory_structure(root, max_depth=3):
    """
    Print a small directory listing to help diagnose path problems.
    """

    print("\nDirectory structure for diagnosis:")

    for current, dirs, files in os.walk(root):
        relative = os.path.relpath(current, root)
        depth = 0 if relative == "." else relative.count(os.sep) + 1

        if depth > max_depth:
            dirs[:] = []
            continue

        print(
            "  " * depth,
            os.path.basename(current) or current,
            "| Subfolders:", dirs[:8],
            "| Example files:", files[:5]
        )


def discover_image_mask_pairs(dataset_root):
    """
    Recursively discover image-mask pairs.

    Works with common structures such as:
        STARE/
            train/images/
            train/masks/
            test/images/
            test/masks/

    Also supports common original STARE structures such as:
        STARE/
            images/
            labels-ah/

    or:
        STARE/
            Original Images/
            Labels ah/
    """

    stare_root = find_stare_root(dataset_root)

    print("\nSearching for STARE images and masks...")
    print("Selected directory:", stare_root)

    all_files = []

    for current, dirs, files in os.walk(stare_root):
        for filename in files:
            ext = os.path.splitext(filename)[1].lower()

            if ext in IMAGE_EXTENSIONS:
                all_files.append(os.path.join(current, filename))

    all_files.sort()

    image_files = [
        path for path in all_files
        if not is_mask_file(path)
    ]

    mask_files = [
        path for path in all_files
        if is_mask_file(path)
    ]

    print("Candidate image files:", len(image_files))
    print("Candidate mask files:", len(mask_files))

    # Build a dictionary of available masks.
    # If several masks match an image, prefer the AH annotation.

    mask_index = {}

    for mask_path in sorted(mask_files, key=mask_priority):
        key = canonical_key(mask_path)

        if not key:
            continue

        if key not in mask_index:
            mask_index[key] = mask_path

    # Match each image to a mask with the same canonical key.

    pairs = []
    used_image_keys = set()

    for image_path in image_files:
        key = canonical_key(image_path)

        if not key or key in used_image_keys:
            continue

        if key in mask_index:
            pairs.append((image_path, mask_index[key]))
            used_image_keys.add(key)

    pairs.sort(key=lambda pair: pair[0])

    print("Successfully matched image-mask pairs:", len(pairs))

    if not pairs:
        display_directory_structure(stare_root)

        raise ValueError(
            "\nNo image-mask pairs were found.\n"
            "The DataLoader error is caused by an empty dataset.\n"
            "Check the displayed directory structure and filenames."
        )

    print("\nExample matched pairs:")

    for image_path, mask_path in pairs[:5]:
        print("Image:", os.path.relpath(image_path, stare_root))
        print("Mask :", os.path.relpath(mask_path, stare_root))
        print("-" * 50)

    return stare_root, pairs


# ============================================================
# 4. TRAIN-TEST SPLITTING
# ============================================================

TRAIN_DIR_NAMES = {
    "train",
    "training",
    "trainset",
    "tarin"
}

TEST_DIR_NAMES = {
    "test",
    "testing",
    "testset",
    "val",
    "valid",
    "validation"
}


def get_split_label(path, root):
    """
    Check whether a path is inside an existing train/test folder.
    """

    relative = os.path.relpath(path, root)
    components = relative.split(os.sep)

    for component in components:
        normalized = re.sub(
            r"[^a-z0-9]",
            "",
            component.lower()
        )

        if (
            normalized in TRAIN_DIR_NAMES
            or normalized.startswith("train")
        ):
            return "train"

        if (
            normalized in TEST_DIR_NAMES
            or normalized.startswith("test")
            or normalized.startswith("valid")
            or normalized == "val"
        ):
            return "test"

    return None


def split_image_mask_pairs(stare_root, pairs):
    """
    Use existing train/test directories when both splits are present.
    Otherwise create a reproducible 80:20 split.
    """

    train_pairs = []
    test_pairs = []

    for image_path, mask_path in pairs:
        image_split = get_split_label(image_path, stare_root)
        mask_split = get_split_label(mask_path, stare_root)

        labels = {
            label for label in (image_split, mask_split)
            if label is not None
        }

        # Avoid accidentally pairing a training image with a test mask.
        if len(labels) > 1:
            continue

        label = next(iter(labels), None)

        if label == "train":
            train_pairs.append((image_path, mask_path))
        elif label == "test":
            test_pairs.append((image_path, mask_path))

    if train_pairs and test_pairs:
        print("\nUsing the existing train/test split.")
        print("Training pairs:", len(train_pairs))
        print("Testing pairs :", len(test_pairs))

        return train_pairs, test_pairs

    # No complete predefined split was found.
    # Split the full set deterministically instead.

    if len(pairs) < 2:
        raise ValueError(
            f"Only {len(pairs)} image-mask pair(s) were found. "
            "At least two are required for a train/test split."
        )

    shuffled_pairs = list(pairs)
    random.Random(SEED).shuffle(shuffled_pairs)

    number_of_test_images = max(
        1,
        int(round(len(shuffled_pairs) * 0.20))
    )

    number_of_test_images = min(
        number_of_test_images,
        len(shuffled_pairs) - 1
    )

    test_pairs = shuffled_pairs[:number_of_test_images]
    train_pairs = shuffled_pairs[number_of_test_images:]

    print("\nNo complete predefined train/test split found.")
    print("Created a reproducible 80:20 split.")
    print("Training pairs:", len(train_pairs))
    print("Testing pairs :", len(test_pairs))

    return train_pairs, test_pairs


# ============================================================
# 5. DATASET CLASS
# ============================================================

class StareDataset(Dataset):

    def __init__(self, pairs, mode="train"):
        self.pairs = pairs
        self.mode = mode

        if len(self.pairs) == 0:
            raise ValueError(
                f"The {mode} dataset is empty. "
                "Check image-mask matching and the dataset path."
            )

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        image_path, mask_path = self.pairs[idx]

        # Read the RGB fundus image.

        img = cv2.imread(image_path, cv2.IMREAD_COLOR)

        if img is None:
            raise RuntimeError(
                f"Unable to read image: {image_path}"
            )

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(
            img,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_LINEAR
        )

        # CLAHE preprocessing.

        img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])

        img = cv2.cvtColor(
            img_lab,
            cv2.COLOR_LAB2RGB
        )

        # Read the binary vessel mask.

        mask = cv2.imread(
            mask_path,
            cv2.IMREAD_GRAYSCALE
        )

        if mask is None:
            raise RuntimeError(
                f"Unable to read mask: {mask_path}"
            )

        mask = cv2.resize(
            mask,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

        # Convert image into C x H x W format.

        img = img.astype(np.float32) / 255.0

        img = np.transpose(img, (2, 0, 1))
        img = np.ascontiguousarray(img)

        # Binary ground truth: background = 0, vessels = 1.

        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)
        mask = np.ascontiguousarray(mask)

        return (
            torch.from_numpy(img),
            torch.from_numpy(mask)
        )


# ============================================================
# 6. ATTENTION GATE
# ============================================================

class AttentionGate(nn.Module):

    def __init__(self, F_g, F_l, F_int):

        super().__init__()

        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1),
            nn.BatchNorm2d(F_int)
        )

        self.W_l = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):

        g1 = self.W_g(g)
        x1 = self.W_l(x)

        attention = self.psi(
            self.relu(g1 + x1)
        )

        return x * attention


# ============================================================
# 7. CONVOLUTION BLOCK
# ============================================================

class ConvBlock(nn.Module):

    def __init__(self, in_ch, out_ch):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


# ============================================================
# 8. ATTENTION-LMFRNET MODEL
# ============================================================

class Attention_LMFRNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.maxpool = nn.MaxPool2d(2)

        # Encoder.

        self.enc1 = ConvBlock(3, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)

        # Decoder level 4.

        self.up4 = nn.ConvTranspose2d(
            512, 256, kernel_size=2, stride=2
        )

        self.att4 = AttentionGate(256, 256, 128)
        self.dec4 = ConvBlock(512, 256)

        # Decoder level 3.

        self.up3 = nn.ConvTranspose2d(
            256, 128, kernel_size=2, stride=2
        )

        self.att3 = AttentionGate(128, 128, 64)
        self.dec3 = ConvBlock(256, 128)

        # Decoder level 2.

        self.up2 = nn.ConvTranspose2d(
            128, 64, kernel_size=2, stride=2
        )

        self.att2 = AttentionGate(64, 64, 32)
        self.dec2 = ConvBlock(128, 64)

        # One output channel for binary vessel segmentation.

        self.final = nn.Conv2d(
            64, 1, kernel_size=1
        )

    def forward(self, x):

        # Encoder.

        e1 = self.enc1(x)
        e2 = self.enc2(self.maxpool(e1))
        e3 = self.enc3(self.maxpool(e2))
        e4 = self.enc4(self.maxpool(e3))

        # Decoder level 4.

        d4 = self.up4(e4)
        x4 = self.att4(d4, e3)

        d4 = self.dec4(
            torch.cat((x4, d4), dim=1)
        )

        # Decoder level 3.

        d3 = self.up3(d4)
        x3 = self.att3(d3, e2)

        d3 = self.dec3(
            torch.cat((x3, d3), dim=1)
        )

        # Decoder level 2.

        d2 = self.up2(d3)
        x2 = self.att2(d2, e1)

        d2 = self.dec2(
            torch.cat((x2, d2), dim=1)
        )

        return self.final(d2)


# ============================================================
# 9. FOCAL DICE LOSS
# ============================================================

class FocalDiceLoss(nn.Module):

    def __init__(self, alpha=0.8, gamma=2.0):

        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):

        # Focal binary cross-entropy.

        bce = F.binary_cross_entropy_with_logits(
            inputs,
            targets,
            reduction="none"
        )

        pt = torch.exp(-bce)

        focal = (
            self.alpha
            * (1.0 - pt) ** self.gamma
            * bce
        )

        focal_loss = focal.mean()

        # Dice loss computed independently for each image.

        probs = torch.sigmoid(inputs)

        dims = (1, 2, 3)

        intersection = (probs * targets).sum(dim=dims)

        denominator = (
            probs.sum(dim=dims)
            + targets.sum(dim=dims)
        )

        dice_score = (
            2.0 * intersection + 1e-7
        ) / (
            denominator + 1e-7
        )

        dice_loss = (1.0 - dice_score).mean()

        return focal_loss + dice_loss


# ============================================================
# 10. TRAINING FUNCTION
# ============================================================

def train_model():

    # Discover real image-mask pairs.

    stare_root, all_pairs = discover_image_mask_pairs(
        BASE_PATH
    )

    train_pairs, test_pairs = split_image_mask_pairs(
        stare_root,
        all_pairs
    )

    train_dataset = StareDataset(
        train_pairs,
        mode="train"
    )

    test_dataset = StareDataset(
        test_pairs,
        mode="test"
    )

    print("\nDataset verification")
    print("-" * 40)
    print("Total pairs :", len(all_pairs))
    print("Train pairs :", len(train_dataset))
    print("Test pairs  :", len(test_dataset))
    print("-" * 40)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda")
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda")
    )

    # Initialize model.

    model = Attention_LMFRNet().to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    criterion = FocalDiceLoss()

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    print("\nStarting training on:", device)

    # Training loop.

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0.0

        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{EPOCHS}"
        )

        for img, mask in progress_bar:

            img = img.to(
                device,
                non_blocking=True
            )

            mask = mask.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            outputs = model(img)

            loss = criterion(outputs, mask)

            loss.backward()

            # Prevent unusually large gradients.

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            running_loss += loss.item()

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        scheduler.step()

        epoch_loss = running_loss / len(train_loader)

        if (epoch + 1) % 5 == 0 or epoch == 0:

            current_lr = optimizer.param_groups[0]["lr"]

            print(
                f"Epoch [{epoch + 1}/{EPOCHS}] "
                f"| Loss: {epoch_loss:.4f} "
                f"| LR: {current_lr:.6g}"
            )

    # Save the trained model.

    model_path = os.path.join(
        OUTPUT_ROOT,
        "attention_lmfrnet_stare.pth"
    )

    torch.save(
        model.state_dict(),
        model_path
    )

    print("\nModel saved to:", model_path)

    return model, test_loader


# ============================================================
# 11. EVALUATION METRICS
# ============================================================

def calculate_metrics(prob, gt_np):

    pred_bin = (prob > 0.5).astype(np.uint8)
    gt_bin = (gt_np > 0.5).astype(np.uint8)

    y_true = gt_bin.ravel()
    y_pred = pred_bin.ravel()
    y_prob = prob.ravel()

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    accuracy = (
        (tp + tn) / (tp + tn + fp + fn + 1e-7)
    )

    dice = (
        (2 * tp) / (2 * tp + fp + fn + 1e-7)
    )

    iou = (
        tp / (tp + fp + fn + 1e-7)
    )

    sensitivity = (
        tp / (tp + fn + 1e-7)
    )

    specificity = (
        tn / (tn + fp + 1e-7)
    )

    # AUC requires both classes to be present.

    if np.unique(y_true).size == 2:
        try:
            auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            auc = np.nan
    else:
        auc = np.nan

    mcc = matthews_corrcoef(y_true, y_pred)

    return {
        "ACC": float(accuracy),
        "Dice": float(dice),
        "IoU": float(iou),
        "Sen": float(sensitivity),
        "Spe": float(specificity),
        "AUC": float(auc),
        "MCC": float(mcc)
    }


# ============================================================
# 12. SAVE VISUAL DASHBOARD
# ============================================================

def save_visual_dashboard(
    img,
    gt,
    prob,
    metrics,
    idx
):

    img_disp = np.transpose(img, (1, 2, 0))

    pred_bin = (prob > 0.5).astype(np.uint8)

    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(np.clip(img_disp, 0, 1))
    plt.title("Original Image - CLAHE Enhanced")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt, cmap="gray", vmin=0, vmax=1)
    plt.title("Ground Truth Vessel Mask")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pred_bin, cmap="gray", vmin=0, vmax=1)
    plt.title(f"Prediction | Dice: {metrics['Dice']:.4f}")
    plt.axis("off")

    stats = " | ".join(
        f"{key}: {value:.4f}"
        for key, value in metrics.items()
    )

    plt.figtext(
        0.5,
        0.02,
        stats,
        ha="center",
        fontsize=9,
        bbox={
            "facecolor": "orange",
            "alpha": 0.2,
            "pad": 5
        }
    )

    plt.tight_layout(rect=[0, 0.08, 1, 1])

    image_path = os.path.join(
        OUTPUT_ROOT,
        f"dashboard_{idx:03d}.png"
    )

    plt.savefig(
        image_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 13. RUN TRAINING AND EVALUATION
# ============================================================

model, test_loader = train_model()

model.eval()

all_metrics = []

print("\nGenerating final metrics and dashboards...")

with torch.no_grad():

    for i, (img, mask) in enumerate(
        tqdm(test_loader, desc="Evaluating")
    ):

        img_device = img.to(device)

        outputs = model(img_device)

        prob = torch.sigmoid(
            outputs
        ).cpu().numpy()[0, 0]

        gt_np = mask.numpy()[0, 0]

        metrics = calculate_metrics(
            prob,
            gt_np
        )

        all_metrics.append(metrics)

        save_visual_dashboard(
            img.numpy()[0],
            gt_np,
            prob,
            metrics,
            i
        )

# ============================================================
# 14. SAVE AND PRINT FINAL RESULTS
# ============================================================

if not all_metrics:
    raise ValueError(
        "No test predictions were produced. "
        "Check the test dataset and image-mask matching."
    )

metric_names = list(all_metrics[0].keys())

average_metrics = {}

for metric_name in metric_names:

    values = np.array(
        [row[metric_name] for row in all_metrics],
        dtype=np.float64
    )

    valid_values = values[np.isfinite(values)]

    if len(valid_values) > 0:
        average_metrics[metric_name] = float(
            np.mean(valid_values)
        )
    else:
        average_metrics[metric_name] = float("nan")

# Save per-image metrics.

per_image_path = os.path.join(
    OUTPUT_ROOT,
    "per_image_metrics.csv"
)

with open(
    per_image_path,
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=metric_names
    )

    writer.writeheader()
    writer.writerows(all_metrics)

# Save average metrics.

summary_path = os.path.join(
    OUTPUT_ROOT,
    "summary_metrics.csv"
)

with open(
    summary_path,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow(["METRIC", "VALUE"])

    for key, value in average_metrics.items():
        writer.writerow([key, value])

# Print summary.

print("\n" + "=" * 44)
print(f"{'METRIC':<15} | {'VALUE':<12}")
print("-" * 44)

for key, value in average_metrics.items():
    print(f"{key:<15} | {value:.4f}")

print("=" * 44)

print("\nFiles saved in:", OUTPUT_ROOT)
print("Trained model       : attention_lmfrnet_stare.pth")
print("Per-image metrics   : per_image_metrics.csv")
print("Summary metrics     : summary_metrics.csv")
print("Prediction images   : dashboard_000.png, etc.")

Device: cuda
Dataset root: /kaggle/input/datasets/abhinav2807/retinal66
Output folder: /kaggle/working/stare_results

Searching for STARE images and masks...
Selected directory: /kaggle/input/datasets/abhinav2807/retinal66/DRIVE CHASE STARE DATASET/STARE
Candidate image files: 40
Candidate mask files: 40
Successfully matched image-mask pairs: 20

Example matched pairs:
Image: tarin/images/im0162.ppm
Mask : tarin/masks/im0162.ah.ppm
--------------------------------------------------
Image: tarin/images/im0163.ppm
Mask : tarin/masks/im0163.ah.ppm
--------------------------------------------------
Image: tarin/images/im0235.ppm
Mask : tarin/masks/im0235.ah.ppm
--------------------------------------------------
Image: tarin/images/im0236.ppm
Mask : tarin/masks/im0236.ah.ppm
--------------------------------------------------
Image: tarin/images/im0239.ppm
Mask : tarin/masks/im0239.ah.ppm
--------------------------------------------------

Using the existing train/test split.
Training pairs:

Epoch 1/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch [1/100] | Loss: 0.9183 | LR: 0.000199951


Epoch 2/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 3/100:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 46/100:   0%|          | 0/5 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a0d37ca4400>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.13/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/usr/lib/python3.13/multiprocessing/popen_fork.py", line 41, in wait
    if not wait([self.sentinel], timeout):
  File "/usr/lib/python3.13/multiprocessing/connection.py", line 1174, in wait
    ready = selector.select(timeout)
  File "/usr/lib/python3.13/selectors.py", line 398, in select
    fd_event_list = self._selector.poll(timeout)
KeyboardInterrupt: 


Epoch 47/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 48/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 49/100:   0%|          | 0/5 [00:00<?, ?it/s]